# LASSO -- universo local exclusivo (EPH Gran La Plata, sin ICG ni macro nacional)


Corre los cuatro objetivos de modelado que conviven en el repo (`delta_v`, `delta_participacion_pct`, `delta_voto_exit_total_pct` construida localmente por D22, `magnitud_desplazamiento_ideologico`) en los tres niveles.

### Paso 1 -- Carga del panel y universo de columnas EPH candidatas

In [1]:
import sys
import pandas as pd
import numpy as np

general_path = "/workspaces/analisis-politica-economia/"
data_path = f"{general_path}data/tfi_data/"
sys.path.insert(0, f"{general_path}/src")

import ml_models
from ml_models.cargar_panel import cargar_panel, columnas_candidatas
import importlib
import ml_models.lasso
from ml_models.lasso import *
importlib.reload(ml_models.lasso)

NIVELES = ["municipal", "provincial", "nacional"]
paneles = {nivel: cargar_panel(nivel, f"{data_path}panel_ventanas.csv") for nivel in NIVELES}

for nivel, df in paneles.items():
    print(f"{nivel}: {df.shape[0]} filas x {df.shape[1]} columnas")

municipal: 12 filas x 151 columnas
provincial: 12 filas x 151 columnas
nacional: 12 filas x 151 columnas


In [2]:
PREFIJOS_EPH = [
    "tasa_informalidad", "pct_sin_cobertura_salud", "hacinamiento_medio",
    "pct_hogares_ayuda_social_gobierno", "pct_hogares_prestamo_bancario",
    "pct_hogares_vendio_pertenencias",
]

TARGETS = ["delta_v", "delta_participacion_pct", "delta_voto_exit_total_pct", "magnitud_desplazamiento_ideologico"]

for nivel in NIVELES:
    df = paneles[nivel]
    if "delta_voto_exit_total_pct" not in df.columns:
        df["delta_voto_exit_total_pct"] = df["delta_voto_exit_ausentismo_pct"] + df["delta_voto_exit_blanco_nulo_pct"]

cols_eph_por_nivel = {}
for nivel in NIVELES:
    df = paneles[nivel]
    cols = columnas_candidatas(df, excluir_adicional=TARGETS + ["delta_voto_exit_ausentismo_pct", "delta_voto_exit_blanco_nulo_pct"])
    cols_eph_por_nivel[nivel] = [c for c in cols if any(c.startswith(p) for p in PREFIJOS_EPH)]
    print(f"{nivel}: {len(cols_eph_por_nivel[nivel])} columnas EPH candidatas, N={len(df)}")

municipal: 25 columnas EPH candidatas, N=12
provincial: 25 columnas EPH candidatas, N=12
nacional: 25 columnas EPH candidatas, N=12


/tmp/ipykernel_2754/279977650.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["delta_voto_exit_total_pct"] = df["delta_voto_exit_ausentismo_pct"] + df["delta_voto_exit_blanco_nulo_pct"]


### Paso 2 -- Sub-selección: colapsar clusters redundantes (umbral 0.90)

Igual criterio que el resto de los notebooks: la reducción depende solo de la colinealidad entre predictores, no del target, así que se corre una vez por nivel y se reutiliza para los cuatro objetivos. `ORDEN_SUFIJO` prioriza nivel/tendencia de corto plazo por sobre volatilidad/cobertura al elegir representante de cada cluster; no hay `PRIORIDAD_TEORICA` entre variables porque acá todas son EPH, ninguna tiene prioridad teórica declarada sobre las demás en el registro.

In [3]:
UMBRAL_REDUNDANCIA = 0.90
ORDEN_SUFIJO = ["_nivel_vc", "_delta_nivel", "_final_vc", "_pendiente_vc", "_volatilidad_vc", "_cobertura_parcial"]

cols_finales_por_nivel = {}
for nivel in NIVELES:
    df = paneles[nivel]
    cols_eph = cols_eph_por_nivel[nivel]
    corr = df[cols_eph].corr(method="pearson")
    clusters = encontrar_redundantes(corr, UMBRAL_REDUNDANCIA)
    representantes = [elegir_representante(c, df=df, orden_sufijo=ORDEN_SUFIJO) for c in clusters]
    cols_finales_por_nivel[nivel] = sorted(representantes)
    print(f"{nivel}: {len(cols_eph)} -> {len(representantes)} tras colapsar clusters (>= |{UMBRAL_REDUNDANCIA}|)")
    for c in clusters:
        if len(c) > 1:
            rep = elegir_representante(c, df=df, orden_sufijo=ORDEN_SUFIJO)
            print(f"    cluster: {sorted(c)} -> representante: {rep}")

municipal: 25 -> 19 tras colapsar clusters (>= |0.9|)
    cluster: ['hacinamiento_medio_cobertura_parcial', 'pct_hogares_ayuda_social_gobierno_cobertura_parcial', 'pct_hogares_prestamo_bancario_cobertura_parcial', 'pct_hogares_vendio_pertenencias_cobertura_parcial', 'pct_sin_cobertura_salud_cobertura_parcial'] -> representante: hacinamiento_medio_cobertura_parcial
    cluster: ['tasa_informalidad_delta_nivel', 'tasa_informalidad_pendiente_vl'] -> representante: tasa_informalidad_pendiente_vl
    cluster: ['tasa_informalidad_final_vc', 'tasa_informalidad_nivel_vc'] -> representante: tasa_informalidad_nivel_vc
provincial: 25 -> 19 tras colapsar clusters (>= |0.9|)
    cluster: ['hacinamiento_medio_cobertura_parcial', 'pct_hogares_ayuda_social_gobierno_cobertura_parcial', 'pct_hogares_prestamo_bancario_cobertura_parcial', 'pct_hogares_vendio_pertenencias_cobertura_parcial', 'pct_sin_cobertura_salud_cobertura_parcial'] -> representante: hacinamiento_medio_cobertura_parcial
    cluster: ['t

### Paso 3 -- LASSO LOO-CV (1-SE) + ajuste final, por objetivo y nivel


In [4]:
resumen_filas = []

for target in TARGETS:
    print(f"\n{'#'*70}\nOBJETIVO: {target}\n{'#'*70}")
    for nivel in NIVELES:
        df = paneles[nivel]
        cols = cols_finales_por_nivel[nivel]
        X, y = construir_Xy_final(nivel, cols, paneles, target=target)

        n, p = X.shape
        if n < 4:
            print(f"  [{nivel}] N={n} insuficiente para LOO-CV, se salta")
            continue

        baseline = baseline_trivial_loocv(y)
        resultado_cv = lasso_loocv_manual(X, y, n_alphas=50)
        alpha_1se = resultado_cv["alpha_1se"]
        idx_1se = np.argmin(np.abs(resultado_cv["alphas"] - alpha_1se))
        mse_1se = resultado_cv["mean_mse"][idx_1se]

        beta_final = ajustar_final(X, y, alpha_1se)
        activos = beta_final[beta_final != 0].sort_values(key=abs, ascending=False)

        print(f"\n  [{nivel}] N={n}, P={p}  |  MSE trivial(LOO)={baseline:.3f}  MSE LASSO(1-SE)={mse_1se:.3f}  alpha_1se={alpha_1se:.4f}")
        if len(activos):
            print(f"    Coeficientes activos (escala estandarizada): {dict(activos.round(3))}")
        else:
            print("    Ningún coeficiente sobrevive alpha_1se (modelo nulo -- predice la media)")

        frac_seleccion = {}
        if len(activos):
            estab = estabilidad_seleccion(nivel, alpha_1se, df, cols, target, X, y)
            frac_seleccion = (estab[activos.index] != 0).mean().round(2).to_dict()
            print(f"    Estabilidad (frac. de corridas leave-one-transition-out con coef != 0): {frac_seleccion}")

        resumen_filas.append({
            "objetivo": target, "nivel": nivel, "N": n, "P": p,
            "mse_trivial": round(baseline, 3), "mse_lasso_1se": round(mse_1se, 3),
            "mejora_pct": round(100 * (1 - mse_1se / baseline), 1) if baseline else None,
            "n_activos": len(activos),
            "variables_activas": "; ".join(activos.index) if len(activos) else "(ninguna)",
        })


######################################################################
OBJETIVO: delta_v
######################################################################
[municipal] excluye 2 fila(s) por NaN: ['municipal_2001_2003', 'municipal_2003_2005']

  [municipal] N=10, P=19  |  MSE trivial(LOO)=251.482  MSE LASSO(1-SE)=257.115  alpha_1se=8.7922
    Ningún coeficiente sobrevive alpha_1se (modelo nulo -- predice la media)
[provincial] excluye 2 fila(s) por NaN: ['provincial_2001_2003', 'provincial_2003_2005']

  [provincial] N=10, P=19  |  MSE trivial(LOO)=245.537  MSE LASSO(1-SE)=325.631  alpha_1se=6.0758
    Coeficientes activos (escala estandarizada): {'hacinamiento_medio_cobertura_parcial': np.float64(0.0)}
    Estabilidad (frac. de corridas leave-one-transition-out con coef != 0): {'hacinamiento_medio_cobertura_parcial': 0.4}
[nacional] excluye 2 fila(s) por NaN: ['nacional_2001_2003', 'nacional_2003_2005']

  [nacional] N=10, P=19  |  MSE trivial(LOO)=190.420  MSE LASSO(1-SE)=235.281

### Paso 4 -- Resumen

In [5]:
resumen = pd.DataFrame(resumen_filas)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)
resumen

,objetivo,nivel,N,P,mse_trivial,mse_lasso_1se,mejora_pct,n_activos,variables_activas
0,delta_v,municipal,10,19,251.482,257.115,-2.2,0,(ninguna)
1,delta_v,provincial,10,19,245.537,325.631,-32.6,1,hacinamiento_medio_cobertura_parcial
2,delta_v,nacional,10,19,190.420,235.281,-23.6,0,(ninguna)
3,delta_participacion_pct,municipal,10,19,26.018,36.483,-40.2,0,(ninguna)
4,delta_participacion_pct,provincial,10,19,27.628,38.677,-40.0,1,tasa_informalidad_volatilidad_vl
5,delta_participacion_pct,nacional,10,19,32.626,44.045,-35.0,0,(ninguna)
6,delta_voto_exit_total_pct,municipal,10,19,16.999,19.019,-11.9,0,(ninguna)
7,delta_voto_exit_total_pct,provincial,10,19,21.243,24.001,-13.0,0,(ninguna)
8,delta_voto_exit_total_pct,nacional,10,19,57.599,68.375,-18.7,0,(ninguna)
9,magnitud_desplazamiento_ideologico,municipal,10,19,0.156,0.162,-4.0,0,(ninguna)


### Notas de lectura

- 10 de las 12 combinaciones nivel×objetivo empeoran respecto del baseline trivial (ver `mejora_pct` en la tabla de arriba) -- con este universo exclusivamente local, el modelo no encuentra señal utilizable ahí.
- Única excepción real (mejora + coeficientes con estabilidad alta): `magnitud_desplazamiento_ideologico`, en **provincial** (N=10, mejora 26.1% sobre el trivial; `tasa_informalidad_pendiente_vc` con estabilidad 1.0) y en **nacional** (N=10, mejora 34.7%; ocho coeficientes activos, `tasa_informalidad_pendiente_vc`/`hacinamiento_medio_delta_nivel` en estabilidad 1.0). `tasa_informalidad` es la familia dominante en los dos niveles.
- **Corrección (2026-09-22) a una nota anterior de esta misma celda**: la corrida exploratoria original (16/09/2026, antes de que D31 extendiera `nacional` a 2001-2025) reportaba acá una segunda excepción -- `delta_v` nacional con N=6 y `pct_hogares_ayuda_social_gobierno_delta_nivel` en estabilidad 1.0 -- que **ya no se sostiene**: con `nacional` en N=10 (mismas 12 transiciones que municipal/provincial, D31), esa combinación pasa a "ningún coeficiente sobrevive alpha_1se" (modelo nulo, ver tabla de arriba). Se documenta como hallazgo frágil que no resistió el aumento de N, no como regresión de esta corrida -- el output de código en sí no cambió por D31 (`04` no usa ninguna variable macro/IPC, ver auditoría D33 en `docs/auditoria_interna/auditoria_ipc_facpce_post_d33.md`), lo que cambió fue el panel de entrada (`panel_ventanas.csv`, D31).
- Lectura sustantiva de la familia `tasa_informalidad`/`pct_hogares_ayuda_social_gobierno`/`pct_hogares_vendio_pertenencias`: el registro de variables las documenta como proxies de estrés económico del hogar, no como señal de generosidad de política pública -- la lectura consistente con esa definición es que el malestar doméstico (informalidad laboral, recurrir a ayuda social o vender pertenencias) se correlaciona con la magnitud del desplazamiento ideológico entre elecciones consecutivas (H2/H3), no con la caída de `delta_v` como sugería la lectura original de esta celda.
- Caveat central: incluso después de colapsar por colinealidad, P=19 sigue por encima de N=10 en los tres niveles -- un régimen más chico que el N=17 de Sinha et al. (2024) ya señalado como frontera de fragilidad para LASSO en este proyecto. Tratar como hipótesis a seguir mirando con próximas elecciones, no como hallazgo cerrado.